# Thesis Progress Summary

This notebook gives a short overview of the current thesis progress:
- datasets used
- preprocessing and multimodal merging
- next possible steps

## 1. Datasets

The thesis uses:
- PaySim for tabular fraud detection
- Document image datasets for image-based fraud detection
- A synthetic merged multimodal dataset for multimodal experiments


In [7]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

print("Current working dir:", Path.cwd())
print("Project root:", project_root)
print("Processed dir exists:", processed_dir.exists())
print("Processed dir:", processed_dir)

Current working dir: c:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\notebook
Project root: c:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis
Processed dir exists: True
Processed dir: c:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\processed


## Final dataset sizes

The final datasets were verified after preprocessing and multimodal pairing.

- PaySim: 4,453,834 train, 636,262 validation, 1,272,524 test
- Image: 6,320 train, 1,419 validation, 2,425 test
- Multimodal: 6,320 train, 1,419 validation, 2,425 test

The final split verification confirmed that there is no overlap between train, validation, and test sets.

In [8]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

# Load final files
paysim_train = pd.read_csv(processed_dir / "paysim_train.csv")
paysim_val = pd.read_csv(processed_dir / "paysim_val.csv")
paysim_test = pd.read_csv(processed_dir / "paysim_test.csv")

image_train = pd.read_csv(processed_dir / "image_train.csv")
image_val = pd.read_csv(processed_dir / "image_val.csv")
image_test = pd.read_csv(processed_dir / "image_test.csv")

mm_train = pd.read_csv(processed_dir / "mm_train.csv")
mm_val = pd.read_csv(processed_dir / "mm_val.csv")
mm_test = pd.read_csv(processed_dir / "mm_test.csv")

# Overlap checks
paysim_overlap_tv = len(set(paysim_train["sample_id"]) & set(paysim_val["sample_id"]))
paysim_overlap_tt = len(set(paysim_train["sample_id"]) & set(paysim_test["sample_id"]))
paysim_overlap_vt = len(set(paysim_val["sample_id"]) & set(paysim_test["sample_id"]))

image_overlap_tv = len(set(image_train["image_path"]) & set(image_val["image_path"]))
image_overlap_tt = len(set(image_train["image_path"]) & set(image_test["image_path"]))
image_overlap_vt = len(set(image_val["image_path"]) & set(image_test["image_path"]))

mm_overlap_tv = len(set(mm_train["img_image_path"]) & set(mm_val["img_image_path"]))
mm_overlap_tt = len(set(mm_train["img_image_path"]) & set(mm_test["img_image_path"]))
mm_overlap_vt = len(set(mm_val["img_image_path"]) & set(mm_test["img_image_path"]))

print("PaySim overlap:")
print("train-val:", paysim_overlap_tv)
print("train-test:", paysim_overlap_tt)
print("val-test:", paysim_overlap_vt)

print("\nImage overlap:")
print("train-val:", image_overlap_tv)
print("train-test:", image_overlap_tt)
print("val-test:", image_overlap_vt)

print("\nMultimodal overlap:")
print("train-val:", mm_overlap_tv)
print("train-test:", mm_overlap_tt)
print("val-test:", mm_overlap_vt)

print("\nFinal multimodal sizes:")
print("train:", mm_train.shape[0])
print("val:", mm_val.shape[0])
print("test:", mm_test.shape[0])

PaySim overlap:
train-val: 0
train-test: 0
val-test: 0

Image overlap:
train-val: 0
train-test: 0
val-test: 0

Multimodal overlap:
train-val: 0
train-test: 0
val-test: 0

Final multimodal sizes:
train: 6320
val: 1419
test: 2425


In [9]:

print("PaySim:")
print(paysim_train.shape, paysim_val.shape, paysim_test.shape)

print("\nImage:")
print(image_train.shape, image_val.shape, image_test.shape)

print("\nMultimodal:")
print(mm_train.shape, mm_val.shape, mm_test.shape)

PaySim:
(4453834, 12) (636262, 12) (1272524, 12)

Image:
(6320, 10) (1419, 10) (2425, 10)

Multimodal:
(6320, 29) (1419, 29) (2425, 29)


In [14]:
summary = pd.DataFrame({
    "Dataset": ["PaySim", "Image", "Multimodal"],
    "Train": [len(paysim_train), len(image_train), len(mm_train)],
    "Validation": [len(paysim_val), len(image_val), len(mm_val)],
    "Test": [len(paysim_test), len(image_test), len(mm_test)]
})

summary

,Dataset,Train,Validation,Test
0,PaySim,4453834,636262,1272524
1,Image,6320,1419,2425
2,Multimodal,6320,1419,2425


## 2. Label distributions


In [11]:
def show_distribution(name, df, target_col):
    print(f"\n{name}")
    print(df[target_col].value_counts())
    print(df[target_col].value_counts(normalize=True) * 100)

show_distribution("PaySim train", paysim_train, "isFraud")
show_distribution("Image train", image_train, "original_label")  
show_distribution("Multimodal train", mm_train, "final_label")


PaySim train
isFraud
0    4448085
1       5749
Name: count, dtype: int64
isFraud
0    99.87092
1     0.12908
Name: proportion, dtype: float64

Image train
original_label
0           2796
forged      2012
attack      1008
bonafide     504
Name: count, dtype: int64
original_label
0           44.240506
forged      31.835443
attack      15.949367
bonafide     7.974684
Name: proportion, dtype: float64

Multimodal train
final_label
0    3300
1    3020
Name: count, dtype: int64
final_label
0    52.21519
1    47.78481
Name: proportion, dtype: float64


## 3. Multimodal merge logic

I first analyzed the tabular and image datasets separately. Then I merged the image datasets into one unified image dataset with binary labels. Finally, I merged this image dataset with PaySim using a synthetic pairing strategy to create the multimodal dataset.

Since the tabular and image datasets do not have natural one-to-one matching samples.

To create the multimodal dataset, I used the following rules:
- samples were matched only within the same split (train, validation, test)
- samples were matched only within the same binary label
- samples were shuffled before pairing
- the final number of multimodal pairs in each split and class was limited by the smaller modality

This created a synthetic but label-aligned multimodal dataset without split leakage.

In [12]:
cols_to_show = [
    "mm_id",
    "split",
    "final_label",
    "tab_step",
    "tab_type",
    "tab_amount",
    "img_image_path",
    "img_image_class",
    "img_source_dataset"
]

mm_train[cols_to_show].head(10)

,mm_id,split,final_label,tab_step,tab_type,tab_amount,img_image_path,img_image_class,img_source_dataset
0,mm_train_1_000783,train,1,434,4,1307156.17,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fantasyid
1,mm_train_1_000544,train,1,427,1,1171751.17,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fmidv
2,mm_train_0_000712,train,0,404,1,82503.76,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,bona_fide,midv
3,mm_train_1_000349,train,1,195,4,594229.77,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fmidv
4,mm_train_0_002873,train,0,397,4,1851728.15,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,bona_fide,midv
5,mm_train_0_000156,train,0,235,3,7111.62,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,bona_fide,midv
6,mm_train_1_001613,train,1,395,1,134297.68,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fmidv
7,mm_train_1_001236,train,1,280,4,268762.37,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fmidv
8,mm_train_1_001721,train,1,221,1,684174.46,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fmidv
9,mm_train_1_002320,train,1,498,4,95064.86,C:\Users\Admin\Desktop\thesis\multimodal-fraud...,forged,fmidv


## 4. Experiment setup

Current planned experiments:
1. Tabular-only model on full PaySim
2. Tabular-only model on merged subset
3. Image-only model on full image dataset
4. Image-only model on merged subset
5. Multimodal model on merged subset

## 5. Current observations

- The multimodal dataset was created successfully.
- The merged dataset contains both tabular PaySim features and image paths.
- The multimodal dataset is smaller than the original datasets because pairing is limited by the smaller class size in each split.
- A fair comparison is possible on the merged subset.

## 6. Next steps

The next step is to train and compare:
- full tabular baseline
- merged tabular baseline
- full image baseline
- merged image baseline
- merged multimodal baseline

In [13]:
import pandas as pd
from pathlib import Path

processed_dir = Path(r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\data\processed")

files = {
    "MIDV": "midv_image_manifest.csv",
    "FantasyID": "fantasyid_image_manifest.csv",
    "FMIDV": "fmidv_image_manifest.csv",
    "PaySim train": "paysim_train.csv",
}

for name, fname in files.items():
    print("\n" + "="*60)
    print(name)
    print("="*60)

    df = pd.read_csv(processed_dir / fname)
    print("Columns:", df.columns.tolist())

    if "image_class" in df.columns:
        print("\nAssigned image_class:")
        print(df["image_class"].value_counts(dropna=False))

    if "original_label" in df.columns:
        print("\nOriginal label:")
        print(df["original_label"].value_counts(dropna=False))

    if "isFraud" in df.columns:
        print("\nOriginal PaySim target:")
        print(df["isFraud"].value_counts(dropna=False))


MIDV
Columns: ['image_path', 'source_dataset', 'image_class', 'group_key', 'doc_type', 'source_type', 'file_name', 'split_source', 'original_label', 'relative_path', 'file_stem', 'suffix']

Assigned image_class:
image_class
bona_fide    4000
Name: count, dtype: int64

Original label:
original_label
0    4000
Name: count, dtype: int64

FantasyID
Columns: ['image_path', 'source_dataset', 'image_class', 'group_key', 'doc_type', 'source_type', 'file_name', 'split_source', 'original_label', 'relative_path', 'file_stem', 'suffix', 'capture_type', 'attack_type', 'width', 'height', 'is_synthetic', 'final_label']

Assigned image_class:
image_class
forged       2351
bona_fide     933
Name: count, dtype: int64

Original label:
original_label
attack      2351
bonafide     933
Name: count, dtype: int64

FMIDV
Columns: ['image_path', 'source_dataset', 'image_class', 'group_key', 'doc_type', 'source_type', 'file_name', 'split_source', 'original_label', 'relative_path', 'top_folder', 'sample_id', 'pa